# 11 · Closed-loop 评测：进度、碰撞、可行驶区域与舒适性

只看 imitation loss 或 open-loop displacement error，无法回答“车辆会不会安全地完成任务”。本 notebook 构造一条带障碍物的车道，批量评估候选轨迹的 progress、off-road、collision、acceleration、jerk 和 composite score。

学习目标：

- 设计明确的 episode-level metric contract；
- 区分 safety metric、task metric 和 comfort metric；
- 对 composite score 做权重敏感性分析；
- 理解 NAVSIM / nuPlan 类评测为什么需要 scenario split 和闭环执行。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

rng = np.random.default_rng(33)
dt = 0.1
horizon = 80
time = np.arange(horizon) * dt
lane_width = 3.6
obstacle = np.array([32.0, 0.0])
reference = np.c_[8.0 * time, np.zeros(horizon)]



In [ ]:
def make_candidate(lateral_bias=0.0, noise=0.25, speed_scale=1.0, seed=0):
    local = np.random.default_rng(seed)
    x = 8.0 * speed_scale * time
    y = lateral_bias + 0.25 * np.sin(time / 1.8) + local.normal(0, noise, horizon)
    return np.c_[x, y]

def evaluate_trajectory(traj, obstacle_radius=1.2, lane_width=3.6):
    velocity = np.gradient(traj, dt, axis=0)
    acceleration = np.gradient(velocity, dt, axis=0)
    jerk = np.gradient(acceleration, dt, axis=0)
    distance_to_obstacle = np.linalg.norm(traj - obstacle, axis=1)
    offroad = np.abs(traj[:, 1]) > lane_width / 2
    collision = distance_to_obstacle <= obstacle_radius
    progress = traj[-1, 0] / reference[-1, 0]
    return {
        'progress': float(progress),
        'offroad_rate': float(offroad.mean()),
        'collision': bool(collision.any()),
        'min_obstacle_distance': float(distance_to_obstacle.min()),
        'mean_acceleration': float(np.linalg.norm(acceleration, axis=1).mean()),
        'max_jerk': float(np.linalg.norm(jerk, axis=1).max()),
        'lateral_rmse': float(np.sqrt(np.mean(traj[:, 1] ** 2))),
    }

def batch_candidates(n=180, seed=33, obstacle_radius=1.2):
    local = np.random.default_rng(seed)
    rows, trajectories = [], []
    for i in range(n):
        traj = make_candidate(
            lateral_bias=local.normal(0, 0.9),
            noise=local.uniform(0.05, 0.55),
            speed_scale=local.uniform(0.75, 1.15),
            seed=seed + i,
        )
        trajectories.append(traj)
        rows.append(evaluate_trajectory(traj, obstacle_radius=obstacle_radius))
    return pd.DataFrame(rows), trajectories

metrics, trajectories = batch_candidates()
print(metrics.describe().round(3).to_string())



In [ ]:
def composite_score(metrics, collision_weight=5.0, comfort_weight=0.2):
    score = (
        metrics['progress']
        - 2.0 * metrics['offroad_rate']
        - collision_weight * metrics['collision'].astype(float)
        - comfort_weight * metrics['max_jerk'] / 100.0
    )
    return score

metrics['score'] = composite_score(metrics)
print('collision rate:', metrics['collision'].mean())
print('top score rows:', metrics.sort_values('score', ascending=False).head(3)[['score', 'progress', 'collision', 'max_jerk']].to_dict('records'))



In [ ]:
def show_evaluation(collision_weight=5.0, comfort_weight=0.2, obstacle_radius=1.2):
    rows, trajectories = batch_candidates(obstacle_radius=obstacle_radius)
    rows['score'] = composite_score(rows, collision_weight, comfort_weight)
    selected = rows.sort_values('score', ascending=False).index[0]
    chosen = trajectories[selected]
    summary = rows[['progress', 'offroad_rate', 'collision', 'mean_acceleration', 'max_jerk', 'score']].mean(numeric_only=True)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    for index in rows.sort_values('score', ascending=False).index[:12]:
        ax[0].plot(trajectories[index][:, 0], trajectories[index][:, 1], color='tab:blue', alpha=0.18)
    ax[0].plot(chosen[:, 0], chosen[:, 1], color='black', linewidth=2, label='top composite score')
    circle = plt.Circle(obstacle, obstacle_radius, color='tab:red', alpha=0.3, label='obstacle')
    ax[0].add_patch(circle)
    ax[0].axhspan(-3.6 / 2, 3.6 / 2, color='tab:green', alpha=0.05)
    ax[0].set_title('candidate closed-loop paths')
    ax[0].set_xlabel('x / m')
    ax[0].set_ylabel('y / m')
    ax[0].legend()
    summary.plot(kind='bar', ax=ax[1])
    ax[1].set_title('batch metric summary')
    ax[1].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
    print(summary.round(3).to_string())

interact(
    show_evaluation,
    collision_weight=FloatSlider(min=0.0, max=15.0, step=0.5, value=5.0, description='collision wt'),
    comfort_weight=FloatSlider(min=0.0, max=2.0, step=0.1, value=0.2, description='comfort wt'),
    obstacle_radius=FloatSlider(min=0.5, max=2.5, step=0.1, value=1.2, description='obstacle radius'),
);



### 练习：不要让一个 composite score 隐藏安全失败

- 把 collision_weight 调成 0，观察 top score 是否可能选择碰撞轨迹。
- 对同一批候选报告 collision rate、off-road rate 和 comfort，而不是只报 score。
- 增加 route progress 与 minimum time-to-collision，比较硬约束和软惩罚。
- 把 scenario 按障碍物位置、噪声、速度切成 test slices，避免平均分掩盖 corner case。


In [ ]:
weights = np.linspace(0, 12, 25)
best_collision = []
for weight in weights:
    scores = composite_score(metrics, collision_weight=weight, comfort_weight=0.2)
    best_collision.append(bool(metrics.loc[scores.idxmax(), 'collision']))
plt.plot(weights, best_collision, drawstyle='steps-mid')
plt.yticks([0, 1], ['safe', 'collision'])
plt.xlabel('collision penalty')
plt.title('metric-weight failure mode')
plt.show()



## 完成标准

- 输出一条候选轨迹的场景图和一个 batch metrics 表。
- 报告 progress、collision、off-road、comfort，并说明 composite score 的局限。
- 给出一个“平均分不错但安全失败”的案例。
- 说明如何接入 NAVSIM / nuPlan 的 scenario runner，并保留相同的指标分层。
